In [2]:
import time
import sys
import os

# Add the parent directory to sys.path
notebook_path = os.getcwd()  # Get the current working directory where the notebook is located
parent_directory = os.path.dirname(notebook_path)  # Move to the parent directory
sys.path.append(parent_directory)

# Now you can import config
from config import *
#from Minimal_Model_2D import *
import Minimal_Model_2D as mm2d
import charge_trap_pot_model as ctm 
import time


np.set_printoptions(threshold=sys.maxsize)

In [3]:
NSM = 60            # number of sm x lattice points
M = 60              # should be atleast thrice confinement length
AX = 5              # lattice spacing in nm
AY = 5              # lattice spacing in nm
DELTA = 0.165       # in meV
MEFF_SC = 0.023     # proximitized Ge
MEFF_SM = 0.023     # Ge
NSC = 800
NTOT = 2*NSC + NSM
PHI = 0.5*np.pi 
ALPHAT = 400        # meV-nm^3 # 20 is within perturbative regime for widths atleast up to 100 nm
K_NUM = 8           # No of eigenvalues needed to be found 
#W_MIN = 50

LY = 75           # Width in nm
OMEGA = 2*4*hbm/LY**2/MEFF_SC

print('lengths=', M*AY, LY)
OMEGA = 2*4*hbm/LY**2/MEFF_SC

print('subband energies=', OMEGA/2, 3*OMEGA/2)

print('4t=', 4*hbm/AX**2/MEFF_SM)

MU_SM = 0.8 + OMEGA/2
MU_SC = 0.8 + OMEGA/2

print('mu = ', MU_SM)

ModCT = ctm.Image_charge_model(NTOT, NSM, NSC, M, AX, AY, OMEGA, ALPHAT, DELTA, MEFF_SC, MEFF_SM, MU_SC, MU_SM, t_ox=7, t_sige=48)


lengths= 300 75
subband energies= 1.1789724682017417 3.536917404605225
4t= 265.2688053453919
mu =  1.9789724682017418


In [4]:
#XCT = np.linspace((NSC-10)*AX, (NSC+NSM+10)*AX, 35)
#XCT = np.linspace(0, (NTOT)*AX, 35)
XCT = (NSC+NSM/2)*AX

# Define the boundary points
a = -100*AY
b = 0
c = M*AY
d = (M+100)*AY

# Define the number of points between each segment
#n_ab = 50  # number of points between a and b
#n_bc = 101  # number of points between b and c (more points)
#n_cd = 50  # number of points between c and d

n_ab = 5  # number of points between a and b
n_bc = 11  # number of points between b and c (more points)
n_cd = 5  # number of points between c and d




# Create linearly spaced points between each pair
y_ab = np.linspace(a, b, n_ab, endpoint=False)
y_bc = np.linspace(b, c, n_bc, endpoint=False)
y_cd = np.linspace(c, d, n_cd, endpoint=True)

# Concatenate all points
YCT = np.concatenate((y_ab, y_bc, y_cd))

#print(YCT)

print('X co-ordinate of charge trap =', XCT )

ZCT = 7
ZIM = 0   ### immaterial in img charge model 

Ep = np.empty((YCT.shape[0],K_NUM))

for i in range(YCT.shape[0]):

    H = ModCT.H_sys(PHI, 'cu', XCT, YCT[i], ZCT, ZIM, f_ct=1)

    E = eigsh(H, k=K_NUM, sigma=0, return_eigenvectors=False )

    Es = np.sort(E)

    Ep[i,:] = Es

    #if i %10 ==0 :
    if i %1 ==0 :
        print(Ep[i,:])

np.save('cubic_E_vs_yct_image_chrg_model.npy', Ep)

X co-ordinate of charge trap = 4150.0
[-0.16624607 -0.16621172 -0.07575773 -0.07546015  0.07546015  0.07575773
  0.16621172  0.16624607]
[-0.16624596 -0.1662116  -0.07575634 -0.07545878  0.07545878  0.07575634
  0.1662116   0.16624596]
[-0.16624575 -0.16621137 -0.07575361 -0.07545607  0.07545607  0.07575361
  0.16621137  0.16624575]


KeyboardInterrupt: 

In [ ]:
Ep_cu_mm2d = np.load('cubic_E_vs_yct_Zdiff_14_wider_yct_width_75nm_refined.npy')
Ep_cu_ctm = np.load('cubic_E_vs_yct_image_chrg_model.npy')